# Model Creation
## Introduction
This notebook covers the final formatting steps required to match the __[PyPSA](https://pypsa.readthedocs.io/en/latest/)__ input format.

In [1]:
import pandas as pd
import geopandas as gpd
import os
from shapely import Point
import pypsa
import numpy as np

In [2]:
### LOADING DATA
# Setting paths
path = os.getcwd()
results_path = os.path.join(path, 'results')
CODERS_path = os.path.join(path, 'data', 'CODERS')
model_results_path = os.path.join(path, 'model', 'national_model')
os.makedirs(model_results_path, exist_ok=True)

In [3]:
LOAD_SCENARIO = 'CNZ'
REF_YEAR = 2025
BUILD_YEAR = 2030 # Earliest year new projects can come online

In [4]:
# Dicts
gen_type_map = {
    'biomass': 'default_biomass',
    'Biomass': 'default_biomass',
    'MSW': 'default_biomass',
    'NG_CC': 'gas_CC',
    'NG_CG': 'gas_CC', # Need to change
    'NG_SC': 'gas_CT', # Need to confirm
    'biogas': 'default_biogas',
    'coal': 'coal_IGCC', # Need to confirm/change
    'coal_CCS': 'coal_IGCC', # Need to add new type
    'diesel_CT': 'default_diesel',
    'gasoline_CT': 'default_diesel', # Need to add new type?
    'hydro_daily': 'hydro_storage',
    'hydro_monthly': 'hydro_storage',
    'hydro_run': 'hydro',
    'nuclear': 'default_nuclear',
    'oil_CT': 'default_oil', # Need to add new type?
    'oil_ST': 'default_oil',
    'solar_PV': 'default_solar_PV',
    'wind_ons': 'wind_2021'
}

# Reading cluster data
clusters = gpd.read_feather(os.path.join(path, 'results', 'clustered_zone_data.feather')).to_crs('EPSG: 3347')
clusters = clusters[['geometry']]

## Buses
This step creates the buses input file.

In [5]:
buses = pd.read_csv(os.path.join(results_path, 'nearest_nodes.csv'), index_col=0)
buses = buses.drop('node', axis=1).rename_axis('name').rename(columns={'lat':'y', 'lon':'x'})
buses['province'] = buses.index.str[:2]
buses.to_csv(os.path.join(model_results_path, 'buses.csv'))

## Generators
This step combines the hydro, wind/solar and combustion generators into a single file. The time series capacity factor data for wind/solar and hydro are also combined.
### Combustion generators
For combustion generators, the CODERS database is used. The previous renewal year field in the generators table is assumed to be the "start year" for the purposes of calculating the retirement year. The lifetimes are added in the generator pre-processing step in the pypsa_cad model. The units are dropped if they are assumed to retire before 2025, based on the renewal year and assumed lifetime.

In [6]:
# Aggregate the generators based on bus, model, retirement year (rounded) and max_hours
aggregate_gens = False

# Generator Lifetimes
gen_life = {
    'default_biomass': 20,
    'default_biogas': 20,
    'coal_IGCC': 50,
    'coal_pulverized': 50,
    'default_diesel': 25,
    'gas_CC': 25,
    'gas_CT': 25,
    'default_nuclear': 80, # Over-estimated, manually set some of the retirement years below
    'default_SMR': 30,
    'default_oil': 40 
}

In [7]:
generator_data = pd.read_csv(os.path.join(CODERS_path, 'generators.csv'), index_col=0)
combustion_gens_data = generator_data[~generator_data.gen_type.isin(['hydro_run', 'hydro_daily', 'hydro_monthly', 'solar_PV', 'wind_ons'])]

# Function to create geopandas dataframe of points at each node
def mapPoints(data): #given a dataframe with one set of lon/lat co-ords
    geometry = [Point(xy) for xy in zip(data['longitude'], data['latitude'])]
    geo_df = gpd.GeoDataFrame(data, crs = {'init':'EPSG:4326'}, geometry = geometry).to_crs('EPSG: 3347')
    return geo_df
combustion_gens_data = mapPoints(combustion_gens_data)

# Finding cluster
combustion_gens_data = combustion_gens_data.sjoin_nearest(clusters, how='left')
combustion_gens_data = combustion_gens_data.drop('geometry', axis=1)

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


In [30]:
# If the renewal year is greater than reference year but the start year is before reference year, set the renewal to reference year
combustion_gens_data.loc[(combustion_gens_data.previous_renewal_year >= REF_YEAR) & (combustion_gens_data.start_year < REF_YEAR), 'previous_renewal_year'] = REF_YEAR

# Check if entries are unique, some aren't due to co-location, bad naming and different start/close years
aggregation = ['generation_facility_name', 'previous_renewal_year', 'gen_type']
combustion_gens = combustion_gens_data.groupby(aggregation).first()
combustion_gens['p_nom'] = combustion_gens_data.groupby(aggregation)['unit_installed_capacity'].sum()
combustion_gens = combustion_gens.reset_index()

# Map generation types
combustion_gens.loc[:, 'model'] = combustion_gens.loc[:, 'gen_type'].map(gen_type_map)

# lifetimes
combustion_gens['lifetime'] = combustion_gens['model'].map(gen_life)
combustion_gens = combustion_gens[(combustion_gens.previous_renewal_year + combustion_gens.lifetime) >= REF_YEAR]

combustion_gens['retirement_year'] = (5 * np.ceil((combustion_gens['previous_renewal_year'] + combustion_gens['lifetime']) / 5)).astype(int)

if aggregate_gens:
    combustion_gens = combustion_gens.groupby(['cluster', 'model', 'retirement_year']).sum()['p_nom'].reset_index()
    combustion_gens['name'] = combustion_gens.cluster + '_' + combustion_gens.model + '_retire_' + combustion_gens.retirement_year.astype(str)
else:
    combustion_gens = combustion_gens.groupby(['generation_facility_name', 'cluster', 'model', 'retirement_year']).sum()['p_nom'].reset_index()
    combustion_gens['name'] = combustion_gens.generation_facility_name + '_' + combustion_gens.model + '_retire_' + combustion_gens.retirement_year.astype(str)


# Non-extendable for existing generators
combustion_gens['p_nom_extendable'] = False

# Format generators
combustion_gens['lifetime'] = combustion_gens['model'].map(gen_life)
combustion_gens['previous_renewal_year'] = combustion_gens['retirement_year'] - combustion_gens['lifetime']
combustion_gens = combustion_gens.rename(columns={'cluster':'bus', 'previous_renewal_year':'build_year'})
combustion_gens = combustion_gens.set_index('name',drop=True)
combustion_gens = combustion_gens[['bus', 'model', 'p_nom', 'p_nom_extendable', 'build_year', 'lifetime']]

combustion_gens.head()

,bus,model,p_nom,p_nom_extendable,build_year,lifetime
name,,,,,,
150 Mile House_default_biomass_retire_2030,BC_North,default_biomass,6.0,False,2010,20
AB Newsprint_gas_CT_retire_2040,AB_North,gas_CT,63.0,False,2015,25
APF Athabasca_default_biomass_retire_2035,AB_North,default_biomass,131.0,False,2015,20
Air Liquide Scotford_gas_CC_retire_2025,AB_Central,gas_CC,106.0,False,2000,25
Algoma Cogeneration_gas_CC_retire_2035,ON_North,gas_CC,63.0,False,2010,25


#### Optional Generators
Adding expandable natural gas generators at each bus

In [31]:
# Skips locations where gas infrastructure is lacking
no_gas = ['ON_Toronto', 'PE', 'QC_North', 'QC_Central', 'NL_Labrador', 'NL_Newfoundland'] 
optional_gens = {}
for bus in buses.index:
    if bus in no_gas:
        continue
    # Add natural gas generator
    optional_gens[f'{bus}_OPT_gas'] = {
        'bus':bus,
        'model':'gas_CC',
        'group':'ungrouped',
        'p_nom':0,
        'p_nom_extendable':True,
        'build_year':BUILD_YEAR
    }
optional_gens = pd.DataFrame.from_dict(optional_gens).T
optional_gens

,bus,model,group,p_nom,p_nom_extendable,build_year
AB_Central_OPT_gas,AB_Central,gas_CC,ungrouped,0,True,2030
AB_North_OPT_gas,AB_North,gas_CC,ungrouped,0,True,2030
AB_South_OPT_gas,AB_South,gas_CC,ungrouped,0,True,2030
BC_North_OPT_gas,BC_North,gas_CC,ungrouped,0,True,2030
BC_South_OPT_gas,BC_South,gas_CC,ungrouped,0,True,2030
MB_OPT_gas,MB,gas_CC,ungrouped,0,True,2030
NB_East_OPT_gas,NB_East,gas_CC,ungrouped,0,True,2030
NB_West_OPT_gas,NB_West,gas_CC,ungrouped,0,True,2030
NS_East_OPT_gas,NS_East,gas_CC,ungrouped,0,True,2030
NS_West_OPT_gas,NS_West,gas_CC,ungrouped,0,True,2030


#### Wind, solar and hydro
Appending the wind, solar and hydro generators to the generator file, as well as saving the hourly capacity factors.

In [32]:
# Generators
generators = pd.concat([
    combustion_gens,
    optional_gens,
    pd.read_csv(os.path.join(results_path, 'wind_solar_gens.csv'), index_col=0), 
    pd.read_csv(os.path.join(results_path, 'ror_gens.csv'), index_col=0)])
generators.index.name = 'name'
generators.to_csv(os.path.join(model_results_path, 'generators.csv'))

# Capacity Factors
cf_data = pd.concat([
    pd.read_csv(os.path.join(results_path, 'wind_solar_cf.csv'), index_col=0).reset_index(drop=True),
    pd.read_csv(os.path.join(results_path, 'ror_cf.csv'), index_col=0).reset_index(drop=True)], 
    axis=1)
cf_data.index = pd.date_range(f'{REF_YEAR}-01-01', periods=8760, freq='h')
cf_data.to_csv(os.path.join(model_results_path, 'generators-p_max_pu.csv'))

## Storage Units
### Existing storage

In [27]:
# Aggregate the storage based on bus, model, retirement year (rounded) and max_hours
aggregate_storage = False

storage_data = pd.read_csv(os.path.join(CODERS_path, 'storage.csv'), index_col=0)
# Only lithium batteries for now
storage_data = storage_data[storage_data.storage_type == 'storage_lithium']
# Function to create geopandas dataframe of points at each node
storage_data = mapPoints(storage_data)

# Finding cluster
storage_data = storage_data.sjoin_nearest(clusters, how='left')
storage_data = storage_data.drop('geometry',axis=1)

# If the renewal year is greater than REF_YEAR but the start year is before REF_YEAR, set the renewal to REF_YEAR
storage_data.loc[(storage_data.previous_renewal_year >= REF_YEAR) & (storage_data.start_year < REF_YEAR), 'previous_renewal_year'] = REF_YEAR
# Using the renewal year as the start year, unless the start year is greater than REF_YEAR (planned capacity)
storage_data.loc[storage_data.start_year > REF_YEAR, 'previous_renewal_year'] = storage_data.loc[storage_data.start_year > REF_YEAR, 'start_year']

# Lifetimes
storage_data['lifetime'] = 25

# Map storage types
storage_data.loc[:, 'model'] = 'default_liion_battery'

aggregation = ['storage_facility_name', 'previous_renewal_year', 'storage_type']
existing_storage_units = storage_data.groupby(aggregation).first()
existing_storage_units['p_nom'] = storage_data.groupby(aggregation)['storage_capacity'].sum()

# Calculate storage capacity
existing_storage_units['storage_energy'] = storage_data.groupby(aggregation)['storage_energy'].sum()
existing_storage_units['max_hours'] = existing_storage_units.storage_energy.divide(existing_storage_units.storage_capacity).round(2)
existing_storage_units = existing_storage_units.reset_index()

existing_storage_units['retirement_year'] = (5 * np.ceil((existing_storage_units['previous_renewal_year'] + existing_storage_units['lifetime']) / 5)).astype(int)


if aggregate_storage:
    existing_storage_units = existing_storage_units.groupby(['cluster', 'model', 'retirement_year', 'max_hours']).sum()['p_nom'].reset_index()
    existing_storage_units['name'] = existing_storage_units.cluster + '_' + existing_storage_units.model + '_' + existing_storage_units.max_hours.astype(str) + 'h_retire_' + existing_storage_units.retirement_year.astype(str)

else:
    existing_storage_units = existing_storage_units.groupby(['storage_facility_name', 'cluster', 'model', 'retirement_year', 'max_hours']).sum()['p_nom'].reset_index()
    existing_storage_units['name'] = existing_storage_units.storage_facility_name + '_' + existing_storage_units.model + '_' + existing_storage_units.max_hours.astype(str) + 'h_retire_' + existing_storage_units.retirement_year.astype(str)



# Non-extendable for existing storage units
existing_storage_units['p_nom_extendable'] = False
existing_storage_units['state_of_charge_initial'] = 0
existing_storage_units['state_of_charge_initial_per_period'] = False
existing_storage_units['cyclic_state_of_charge_per_period'] = False
existing_storage_units['committable'] = False

# Format storage units
existing_storage_units['lifetime'] = 25
existing_storage_units['previous_renewal_year'] = existing_storage_units['retirement_year'] - existing_storage_units['lifetime']
existing_storage_units = existing_storage_units.rename(columns={'cluster':'bus', 'previous_renewal_year':'build_year'})
existing_storage_units = existing_storage_units.set_index('name',drop=True)
existing_storage_units = existing_storage_units[['bus', 'model', 'p_nom', 'max_hours', 'state_of_charge_initial', 'state_of_charge_initial_per_period', 'cyclic_state_of_charge_per_period', 'p_nom_extendable', 'build_year', 'lifetime']]

existing_storage_units.head()

c:\Users\ndematos\envs\pypsa_canada_py312\Lib\site-packages\pyproj\crs\crs.py:143: FutureWarning: '+init=<authority>:<code>' syntax is deprecated. '<authority>:<code>' is the preferred initialization method. When making the change, be mindful of axis order changes: https://pyproj4.github.io/pyproj/stable/gotchas.html#axis-order-changes-in-proj-6
  in_crs_string = _prepare_from_proj_string(in_crs_string)


,bus,model,p_nom,max_hours,state_of_charge_initial,state_of_charge_initial_per_period,p_nom_extendable,build_year,lifetime
name,,,,,,,,,
903_default_liion_battery_3.8h_retire_2055,ON_North,default_liion_battery,5,3.8,0,False,False,2030,25
Almonte BESS_default_liion_battery_3.8h_retire_2055,ON_East,default_liion_battery,5,3.8,0,False,False,2030,25
Arlen Energy Storage 1_default_liion_battery_4.0h_retire_2055,ON_South,default_liion_battery,19,4.0,0,False,False,2030,25
Bridgewater_default_liion_battery_4.0h_retire_2050,NS_West,default_liion_battery,50,4.0,0,False,False,2025,25
Canaan Rd_default_liion_battery_4.0h_retire_2055,NS_West,default_liion_battery,50,4.0,0,False,False,2030,25


### Optional storage units


In [28]:
no_batteries = ['BC_North', 'MB', 'NL_Labrador', 'QC_North']
optional_storage = {}
for bus in buses.index:
    if bus in no_batteries:
        continue
    # Add optionnal lithium batteries
    optional_storage[f'{bus}_OPT_Battery'] = {
        'bus':bus,
        'model':'default_liion_battery',
        'p_nom':0,
        'p_nom_extendable':True,
        'max_hours':4,
        'state_of_charge_initial_per_period':True,
        'state_of_charge_initial': 0,
        'cyclic_state_of_charge_per_period':False,
        'build_year':BUILD_YEAR,
    }

optional_storage = pd.DataFrame.from_dict(optional_storage).T
optional_storage

,bus,model,p_nom,p_nom_extendable,max_hours,state_of_charge_initial_per_period,state_of_charge_initial,build_year
AB_Central_OPT_Battery,AB_Central,default_liion_battery,0,True,4,True,0,2030
AB_North_OPT_Battery,AB_North,default_liion_battery,0,True,4,True,0,2030
AB_South_OPT_Battery,AB_South,default_liion_battery,0,True,4,True,0,2030
BC_South_OPT_Battery,BC_South,default_liion_battery,0,True,4,True,0,2030
NB_East_OPT_Battery,NB_East,default_liion_battery,0,True,4,True,0,2030
NB_West_OPT_Battery,NB_West,default_liion_battery,0,True,4,True,0,2030
NL_Newfoundland_OPT_Battery,NL_Newfoundland,default_liion_battery,0,True,4,True,0,2030
NS_East_OPT_Battery,NS_East,default_liion_battery,0,True,4,True,0,2030
NS_West_OPT_Battery,NS_West,default_liion_battery,0,True,4,True,0,2030
ON_East_OPT_Battery,ON_East,default_liion_battery,0,True,4,True,0,2030


### Saving hydro storage unit data

In [29]:
storage_units = pd.concat([
    pd.read_csv(os.path.join(results_path, 'reservoir_storage_units.csv'), index_col=0),
    existing_storage_units,
    optional_storage])
storage_units.index.name = 'name'
storage_units.to_csv(os.path.join(model_results_path, 'storage_units.csv'))

inflows = pd.read_csv(os.path.join(results_path, 'storage_units-inflow.csv'), index_col=0).reset_index(drop=True)
inflows.index = pd.date_range(f'{REF_YEAR}-01-01', periods=8760, freq='h')
inflows.to_csv(os.path.join(model_results_path, 'storage_units-inflow.csv'))

## Load data

In [26]:
load_forecast = pd.read_csv(os.path.join(results_path, f'loads_forecast_{LOAD_SCENARIO}.csv'), index_col=0)
load_forecast.index = pd.to_datetime(load_forecast.index)
load_forecast.to_csv(os.path.join(model_results_path, f'loads_forecast_{LOAD_SCENARIO}.csv'))

loads = pd.read_csv(os.path.join(results_path, 'loads.csv'), index_col=0)
loads.to_csv(os.path.join(model_results_path, 'loads.csv'))

## Transmission interfaces

In [15]:
# Read results from line_distances
interface_data = pd.read_csv(os.path.join(results_path, 'cluster_interfaces.csv'))
interface_data = interface_data.rename(columns={'start':'bus0', 'end':'bus1', 'distance':'length', 'capacity_fwd':'p_nom'})

fwd_links = interface_data.copy()
fwd_links['name'] = fwd_links.bus0 + '->' + fwd_links.bus1
fwd_links['p_min_pu'] = 0

bck_links = interface_data[interface_data['capacity_bck'] > 0].copy()
# Flip the buses
bck_links[['bus0', 'bus1']] = bck_links[['bus1', 'bus0']].values
bck_links['p_nom'] = bck_links['capacity_bck']
bck_links['name'] = bck_links.bus0 + '->' + bck_links.bus1
bck_links['p_min_pu'] = 0

interface_data = pd.concat([fwd_links, bck_links], ignore_index=True)
interface_data = interface_data.set_index('name', drop=True)

interface_data['efficiency'] = 0.95
interface_data['build_year'] = 1
interface_data['lifetime'] = 'inf'
interface_data['p_nom_extendable'] = False
interface_data['capital_cost'] = 0
interface_data['carrier'] = 'DC'
interface_data['committable'] = False
interface_data = interface_data.drop(['dijkstra_distance', 'straight_distance', 'capacity_bck'], axis=1)
interface_data.head()

,bus0,bus1,length,p_nom,p_min_pu,efficiency,build_year,lifetime,p_nom_extendable,capital_cost,carrier,committable
name,,,,,,,,,,,,
AB_Central->AB_North,AB_Central,AB_North,437.000000,3588,0,0.95,1,inf,False,0,DC,False
AB_Central->AB_South,AB_Central,AB_South,281.850000,4565,0,0.95,1,inf,False,0,DC,False
AB_South->BC_South,AB_South,BC_South,683.598735,1000,0,0.95,1,inf,False,0,DC,False
AB_South->SK,AB_South,SK,648.898048,150,0,0.95,1,inf,False,0,DC,False
BC_North->BC_South,BC_North,BC_South,634.665000,11546,0,0.95,1,inf,False,0,DC,False


### Expansion options

In [16]:
tx_price = 85.11 # $CAD/km/MW, annualized
optional_links = {}
for name, data in interface_data.iterrows():
    optional_links[f'OPT_Link_{data.bus0}_to_{data.bus1}'] = {
        'bus0':data.bus0,
        'bus1':data.bus1,
        'length':data.length,
        'p_nom':0,
        'p_min_pu':0,
        'efficiency':0.95,
        'build_year':BUILD_YEAR,
        'lifetime':40,
        'p_nom_extendable':True,
        'carrier':'DC',
        'committable':False
    }
    optional_links[f'OPT_Link_{data.bus0}_to_{data.bus1}']['capital_cost'] = round(tx_price/2 * data.length, 2)
optional_links = pd.DataFrame.from_dict(optional_links).T
optional_links

,bus0,bus1,length,p_nom,p_min_pu,efficiency,build_year,lifetime,p_nom_extendable,carrier,committable,capital_cost
OPT_Link_AB_Central_to_AB_North,AB_Central,AB_North,437.0,0,0,0.95,2030,40,True,DC,False,18596.53
OPT_Link_AB_Central_to_AB_South,AB_Central,AB_South,281.85,0,0,0.95,2030,40,True,DC,False,11994.13
OPT_Link_AB_South_to_BC_South,AB_South,BC_South,683.598735,0,0,0.95,2030,40,True,DC,False,29090.54
OPT_Link_AB_South_to_SK,AB_South,SK,648.898048,0,0,0.95,2030,40,True,DC,False,27613.86
OPT_Link_BC_North_to_BC_South,BC_North,BC_South,634.665,0,0,0.95,2030,40,True,DC,False,27008.17
OPT_Link_MB_to_ON_Northwest,MB,ON_Northwest,622.378154,0,0,0.95,2030,40,True,DC,False,26485.3
OPT_Link_MB_to_SK,MB,SK,512.161629,0,0,0.95,2030,40,True,DC,False,21795.04
OPT_Link_NB_East_to_NB_West,NB_East,NB_West,211.24,0,0,0.95,2030,40,True,DC,False,8989.32
OPT_Link_NB_East_to_NS_West,NB_East,NS_West,164.280937,0,0,0.95,2030,40,True,DC,False,6990.98
OPT_Link_NB_East_to_PE,NB_East,PE,73.687412,0,0,0.95,2030,40,True,DC,False,3135.77


In [17]:
interface_data = pd.concat([
    interface_data,
    optional_links
])
interface_data.index.name = 'name'
interface_data.to_csv(os.path.join(model_results_path, 'links.csv'))

# Snapshots File
Needed to load model

In [18]:
snapshots = pd.DataFrame(index=load_forecast.loc[str(REF_YEAR)].index)
snapshots['weightings'] = 1
snapshots.to_csv(os.path.join(model_results_path, 'snapshots.csv'))

## Validation

In [19]:
network = pypsa.Network(model_results_path)
network

INFO:pypsa.network.io:New version 1.2.4 available! (Current: 1.2.2)
INFO:pypsa.network.index:Applying weightings to all columns of `snapshot_weightings`
INFO:pypsa.network.io:Imported network 'Unnamed Network' has buses, generators, links, loads, storage_units


PyPSA Network 'Unnamed Network'
-------------------------------
Components:
 - Bus: 23
 - Generator: 688
 - Link: 106
 - Load: 33
 - StorageUnit: 83
Snapshots: 8760

In [21]:
#Due to incompatibilities with linopy and xarray, pyarrow will be removed here
%pip uninstall -y pyarrow

Found existing installation: pyarrow 24.0.0
Uninstalling pyarrow-24.0.0:
  Successfully uninstalled pyarrow-24.0.0
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.
You can safely remove it manually.
